# GPT

In [19]:
# Mathematical trick for the self-attention
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

## Bag of Words approach

In [3]:
# We want to x[b,t] = mean_{i<==t} x[b,1]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xbow[b,t] = torch.mean(x[b,:t+1], 0)

In [4]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [5]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

## Trick to find out the Bag-Of-Words quicker

In [14]:
torch.manual_seed(42)
#a = torch.ones(3,3)
a = torch.tril(torch.ones(3,3))
a = a/torch.sum(a,1,keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b

print('a==')
print(a)
print('---')
print('b==')
print(b)
print('---')
print('c==')
print(c)


a==
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
---
b==
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
---
c==
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


## Use the trick to find out the average

In [ ]:
wei = torch.tril(torch.ones(8,8))
wei = wei/torch.sum(wei,1,keepdim=True)
xbow2 = wei @ x # (B,T,T) @ (B,T,C) -> (B,T,C)
torch.allclose(xbow, xbow2)

True

In [19]:
# Verison 3
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei,dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

In [1]:
#Version 4: self-attention with one head
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias= False)
k = key(x)       # (B,T,16)
q = query (x)    # (B,T,16)
wei = q @ k.transpose(-2 , -1)    # (B,T,16) @ (B,16,T) = (B,T,T)

tril = torch.tril(torch.ones(T,T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ v

out.shape

NameError: name 'torch' is not defined

In [21]:
wei[0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [5.4949e-01, 4.5051e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [3.0814e-02, 1.8298e-02, 9.5089e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [2.7306e-02, 5.9927e-03, 9.6422e-01, 2.4795e-03, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [1.0356e-01, 8.4526e-02, 2.9864e-01, 4.2126e-01, 9.2020e-02, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [5.5813e-02, 2.6597e-01, 6.9153e-02, 8.4581e-02, 2.7192e-01, 2.5256e-01,
         0.0000e+00, 0.0000e+00],
        [1.1987e-02, 3.8042e-04, 9.6417e-01, 3.2126e-03, 5.0234e-03, 1.4995e-02,
         2.3348e-04, 0.0000e+00],
        [3.9488e-02, 2.8300e-01, 1.5332e-02, 7.4047e-02, 5.5218e-02, 4.9383e-01,
         1.6254e-02, 2.2827e-02]], grad_fn=<SelectBackward>)